In [0]:
%pip install dbtunnel[gradio] aiohttp


In [0]:
dbutils.library.restartPython()

In [0]:
import os
os.environ['DATABRICKS_TOKEN'] = dbutils.secrets.get("hackathon-databricks", "hackathon-databricks")

os.environ['API_ENDPOINT'] = "https://adb-109982197220035.15.azuredatabricks.net/serving-endpoints/insurance_model/invocations"



In [0]:
import itertools
import gradio as gr
import requests
import json
import os  # Added import for os environment variable

def respond(message, history):
    if len(message.strip()) == 0:
        return "ERROR: the question should not be empty"

    print("#### Message #####")
    print(message)

    local_token = os.getenv('DATABRICKS_TOKEN')
    local_endpoint = os.getenv('API_ENDPOINT')

    if local_token is None or local_endpoint is None:
        return "ERROR: missing env variables"

    # Add your API token to the headers
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {local_token}'
    }

    payload = {
        "inputs": [message]
    }

    try:
        response = requests.post(
            local_endpoint, json=payload, headers=headers, timeout=100)

        response_json = response.json()
        print("#### RESPONSE JSON")
        print(response_json)

        response_data = response_json["predictions"][0]  # Updated line to extract the result field
        print(response_data)

    except Exception as error:
        response_data = f"ERROR: status_code: {type(error).__name__}"

    return response_data

# Customize the theme using predefined color shortcuts
theme = gr.themes.Soft()

demo = gr.ChatInterface(
    respond,
    chatbot=gr.Chatbot(
        show_label=False, 
        container=False, 
        show_copy_button=True, 
        bubble_full_width=True,
        height=600  # Increased height of the chatbox
    ),
    textbox=gr.Textbox(
        placeholder="Ask me a question", 
        container=False, 
        scale=10,  # Increased the scale for a larger input area
        elem_id="dynamic-textbox"  # ID for dynamic resizing
    ),
    title="Team REV-Engers - Insurance Comparison AI",
    description="Insurance model created with Databricks AutoML for Databricks Hackathon",
    examples=[
        ["Given the property's age, size, and location, what would the estimated insurance premium be based on historical data for similar assets?"],
    ],
    cache_examples=False,
    theme=theme,
    retry_btn=None,
    undo_btn=None,
    clear_btn="Clear",
)


In [0]:
from dbtunnel import dbtunnel
dbtunnel.gradio(demo).run()